In [ ]:
import subprocess
import json
import os
import time
import sys

# Configuration based on GPT-5.1 advice
CONFIG = {
    "betas": [4.8, 5.0, 5.2],
    "Ls": [8, 10],
    "sweeps": 500,        # Increased for better statistics
    "thermalize": 100,
    "output_dir": "experiment_results"
}

def run_scan():
    print(f"🚀 Starting Defect Scan")
    print(f"Configuration: {json.dumps(CONFIG, indent=2)}")

    os.makedirs(CONFIG["output_dir"], exist_ok=True)
    summary = []

    for L in CONFIG["Ls"]:
        for beta in CONFIG["betas"]:
            print(f"\n▶ Running L={L}, Beta={beta}...")
            outfile = os.path.join(CONFIG["output_dir"], f"results_L{L}_beta{beta}.json")

            # Call the core simulation kernel
            cmd = [
                sys.executable, "analysis_lattice_mc.py",
                "--L", str(L),
                "--beta", str(beta),
                "--sweeps", str(CONFIG["sweeps"]),
                "--therm", str(CONFIG["thermalize"]),
                "--out", outfile
            ]

            try:
                start = time.time()
                subprocess.run(cmd, check=True)
                duration = time.time() - start

                # Read back results
                with open(outfile, 'r') as f:
                    data = json.load(f)

                print(f"  ✅ Done in {duration:.1f}s | Density: {data['avg_density']:.2e}")

                summary.append({
                    "L": L,
                    "beta": beta,
                    "density": data['avg_density'],
                    "defects": data['avg_defects']
                })

            except subprocess.CalledProcessError:
                print(f"  ❌ Simulation failed for L={L}, Beta={beta}")

    # Save summary
    with open("scan_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nScan Complete. Summary saved to scan_summary.json")

if __name__ == "__main__":
    run_scan()

🚀 Starting Defect Scan
Configuration: {
  "betas": [
    4.8,
    5.0,
    5.2
  ],
  "Ls": [
    8,
    10
  ],
  "sweeps": 500,
  "thermalize": 100,
  "output_dir": "experiment_results"
}

▶ Running L=8, Beta=4.8...
  ❌ Simulation failed for L=8, Beta=4.8

▶ Running L=8, Beta=5.0...
  ❌ Simulation failed for L=8, Beta=5.0

▶ Running L=8, Beta=5.2...
  ❌ Simulation failed for L=8, Beta=5.2

▶ Running L=10, Beta=4.8...
  ❌ Simulation failed for L=10, Beta=4.8

▶ Running L=10, Beta=5.0...
  ❌ Simulation failed for L=10, Beta=5.0

▶ Running L=10, Beta=5.2...
  ❌ Simulation failed for L=10, Beta=5.2

Scan Complete. Summary saved to scan_summary.json


In [3]:
# @title Lattice Yang-Mills Defect Scan (JAX)
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np
import time
import json
import os
from datetime import datetime
import matplotlib.pyplot as plt

# --- SU(2) Utils ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

# --- Lattice Setup ---
def init_lattice_cold(L):
    Dim = 4
    NumLinks = L**Dim * Dim
    shape = (L, L, L, L, Dim, 2, 2)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape(shape)

def make_neighbor_table(L):
    sites = np.arange(L**4).reshape((L, L, L, L))
    up = []
    down = []
    for d in range(4):
        up.append(np.roll(sites, -1, axis=d).flatten())
        down.append(np.roll(sites, 1, axis=d).flatten())
    return jnp.array(up), jnp.array(down)

# --- Simulation Kernels (Closure Factory) ---
def make_sweep_fn(L, beta, UP, DOWN):
    Dim = 4
    Nlinks = L**Dim * Dim

    @jit
    def calculate_staples(U_flat, site, mu):
        staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)
        for nu in range(4):
            x = site
            x_plus_mu = UP[mu, x]
            x_plus_nu = UP[nu, x]
            x_minus_nu = DOWN[nu, x]
            x_plus_mu_minus_nu = DOWN[nu, x_plus_mu]

            u1 = U_flat[x_plus_mu, nu]
            u2 = jnp.conjugate(jnp.transpose(U_flat[x_plus_nu, mu]))
            u3 = jnp.conjugate(jnp.transpose(U_flat[x, nu]))
            term_fwd = u1 @ u2 @ u3

            u1_b = jnp.conjugate(jnp.transpose(U_flat[x_plus_mu_minus_nu, nu]))
            u2_b = jnp.conjugate(jnp.transpose(U_flat[x_minus_nu, mu]))
            u3_b = U_flat[x_minus_nu, nu]
            term_bwd = u3_b @ u2_b @ u1_b

            mask = jnp.where(nu == mu, 0.0, 1.0)
            staple_sum += mask * (term_fwd + term_bwd)
        return staple_sum

    @jit
    def update_site_link(key, U_flat, site, mu):
        U = U_flat[site, mu]
        staples = calculate_staples(U_flat, site, mu)

        key, subkey = random.split(key)
        alpha = random.normal(subkey, (3,)) * 0.1
        dU = exp_map_pauli(alpha)
        U_new = dU @ U

        S_old = -0.5 * beta * jnp.real(jnp.trace(U @ staples))
        S_new = -0.5 * beta * jnp.real(jnp.trace(U_new @ staples))

        dS = S_new - S_old
        accept = jnp.log(random.uniform(key)) < -dS
        U_final = jnp.where(accept, U_new, U)

        U_flat = U_flat.at[site, mu].set(U_final)
        return key, U_flat, jnp.where(accept, 1, 0)

    @jit
    def sweep(key, U_flat):
        def body_fun(carry, idx):
            key, U_f = carry
            site = idx // 4
            mu = idx % 4
            key, U_f, acc = update_site_link(key, U_f, site, mu)
            return (key, U_f), acc
        (key, U_flat), accs = lax.scan(body_fun, (key, U_flat), jnp.arange(Nlinks))
        return key, U_flat, jnp.sum(accs)

    return sweep

def make_measure_fn(r_crit):
    @jit
    def get_r(U):
        tr = jnp.real(jnp.trace(U))
        tr = jnp.clip(tr, -2.0, 2.0)
        return jnp.arccos(0.5 * tr)
    @jit
    def measure_defects(U_flat):
        rs = vmap(vmap(get_r))(U_flat)
        defects = rs > r_crit
        return defects, jnp.mean(defects)
    return measure_defects

def run_simulation(L, beta, n_sweeps=100, n_thermalize=50):
    print(f"▶ Running L={L}^4, Beta={beta}...")
    Dim = 4
    Vol = L**Dim
    key = random.PRNGKey(int(time.time()) % (2**31 - 1))
    U_flat = init_lattice_cold(L).reshape((Vol, 4, 2, 2))
    r_crit = 1.9248

    UP, DOWN = make_neighbor_table(L)
    sweep = make_sweep_fn(L, beta, UP, DOWN)
    measure_defects = make_measure_fn(r_crit)

    # Thermalize
    for _ in range(n_thermalize):
        key, U_flat, _ = sweep(key, U_flat)

    # Measure
    measurements = []
    Nlinks = Vol * Dim
    for i in range(n_sweeps):
        key, U_flat, acc = sweep(key, U_flat)
        defects, rho = measure_defects(U_flat)
        measurements.append(float(rho))

    avg_rho = sum(measurements) / len(measurements)
    print(f"  ✅ Density: {avg_rho:.4e}")
    return avg_rho

# --- Main Scan Loop ---
CONFIG = {
    "betas": [4.8, 5.0, 5.2],
    "Ls": [8, 10],
    "sweeps": 200,
    "thermalize": 50
}

results = []
print("🚀 Starting Scan...")

for L in CONFIG["Ls"]:
    for beta in CONFIG["betas"]:
        rho = run_simulation(L, beta, CONFIG["sweeps"], CONFIG["thermalize"])
        results.append({"L": L, "beta": beta, "rho": rho})

print("\n📊 Summary:")
print(json.dumps(results, indent=2))

🚀 Starting Scan...
▶ Running L=8^4, Beta=4.8...
  ✅ Density: 2.1912e-04
▶ Running L=8^4, Beta=5.0...
  ✅ Density: 1.4008e-04
▶ Running L=8^4, Beta=5.2...
  ✅ Density: 1.2787e-04
▶ Running L=10^4, Beta=4.8...
  ✅ Density: 2.5175e-04
▶ Running L=10^4, Beta=5.0...
  ✅ Density: 2.6750e-04
▶ Running L=10^4, Beta=5.2...
  ✅ Density: 1.5662e-04

📊 Summary:
[
  {
    "L": 8,
    "beta": 4.8,
    "rho": 0.0002191162109375
  },
  {
    "L": 8,
    "beta": 5.0,
    "rho": 0.00014007568359375
  },
  {
    "L": 8,
    "beta": 5.2,
    "rho": 0.00012786865234375
  },
  {
    "L": 10,
    "beta": 4.8,
    "rho": 0.00025174999237606246
  },
  {
    "L": 10,
    "beta": 5.0,
    "rho": 0.0002674999931605271
  },
  {
    "L": 10,
    "beta": 5.2,
    "rho": 0.00015662499491554628
  }
]


In [1]:
# @title Lattice Yang-Mills Monopole Scan (JAX)
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np
import time
import json
import os
from datetime import datetime

# --- SU(2) Utils ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

# --- Lattice Setup ---
def init_lattice_cold(L):
    Dim = 4
    NumLinks = L**Dim * Dim
    shape = (L, L, L, L, Dim, 2, 2)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape(shape)

def make_neighbor_table(L):
    sites = np.arange(L**4).reshape((L, L, L, L))
    up = []
    down = []
    for d in range(4):
        up.append(np.roll(sites, -1, axis=d).flatten())
        down.append(np.roll(sites, 1, axis=d).flatten())
    return jnp.array(up), jnp.array(down)

# --- Simulation Kernels ---
def make_sweep_fn(L, beta, UP, DOWN):
    Dim = 4
    Nlinks = L**Dim * Dim

    @jit
    def calculate_staples(U_flat, site, mu):
        staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)
        for nu in range(4):
            x = site
            x_plus_mu = UP[mu, x]
            x_plus_nu = UP[nu, x]
            x_minus_nu = DOWN[nu, x]
            x_plus_mu_minus_nu = DOWN[nu, x_plus_mu]

            u1 = U_flat[x_plus_mu, nu]
            u2 = jnp.conjugate(jnp.transpose(U_flat[x_plus_nu, mu]))
            u3 = jnp.conjugate(jnp.transpose(U_flat[x, nu]))
            term_fwd = u1 @ u2 @ u3

            u1_b = jnp.conjugate(jnp.transpose(U_flat[x_plus_mu_minus_nu, nu]))
            u2_b = jnp.conjugate(jnp.transpose(U_flat[x_minus_nu, mu]))
            u3_b = U_flat[x_minus_nu, nu]
            term_bwd = u3_b @ u2_b @ u1_b

            mask = jnp.where(nu == mu, 0.0, 1.0)
            staple_sum += mask * (term_fwd + term_bwd)
        return staple_sum

    @jit
    def update_site_link(key, U_flat, site, mu):
        U = U_flat[site, mu]
        staples = calculate_staples(U_flat, site, mu)
        key, subkey = random.split(key)
        alpha = random.normal(subkey, (3,)) * 0.1
        dU = exp_map_pauli(alpha)
        U_new = dU @ U
        S_old = -0.5 * beta * jnp.real(jnp.trace(U @ staples))
        S_new = -0.5 * beta * jnp.real(jnp.trace(U_new @ staples))
        dS = S_new - S_old
        accept = jnp.log(random.uniform(key)) < -dS
        U_final = jnp.where(accept, U_new, U)
        U_flat = U_flat.at[site, mu].set(U_final)
        return key, U_flat, jnp.where(accept, 1, 0)

    @jit
    def sweep(key, U_flat):
        def body_fun(carry, idx):
            key, U_f = carry
            site = idx // 4
            mu = idx % 4
            key, U_f, acc = update_site_link(key, U_f, site, mu)
            return (key, U_f), acc
        (key, U_flat), accs = lax.scan(body_fun, (key, U_flat), jnp.arange(Nlinks))
        return key, U_flat, jnp.sum(accs)
    return sweep

# --- Monopole Measurement (DeGrand-Toussaint) ---
def make_measure_fn(L, UP, DOWN):
    Dim = 4
    Vol = L**Dim

    @jit
    def get_abelian_phases(U_flat):
        # Naive Abelian Projection: phase of U_11
        u11 = U_flat[:, :, 0, 0]
        theta = jnp.angle(u11)
        return theta

    @jit
    def measure_monopoles(U_flat):
        theta = get_abelian_phases(U_flat)
        def shift(arr, mu): return arr[UP[mu]]

        total_flux = 0.0
        # Sum over all plaquettes
        for mu in range(4):
            for nu in range(mu + 1, 4):
                t_x_mu = theta[:, mu]
                t_x_nu = theta[:, nu]
                t_x_mu_plus_nu = shift(t_x_mu, nu)
                t_x_nu_plus_mu = shift(t_x_nu, mu)

                flux_raw = t_x_mu + t_x_nu_plus_mu - t_x_mu_plus_nu - t_x_nu
                n_p = jnp.round(flux_raw / (2*jnp.pi))
                total_flux += jnp.sum(jnp.abs(n_p))

        return total_flux / (Vol * 6) # Normalize by total plaquettes

    return measure_monopoles

def run_simulation(L, beta, n_sweeps=100, n_thermalize=50):
    print(f"▶ Running L={L}^4, Beta={beta}...")
    Dim = 4
    Vol = L**Dim
    key = random.PRNGKey(int(time.time()) % (2**31 - 1))
    U_flat = init_lattice_cold(L).reshape((Vol, 4, 2, 2))

    UP, DOWN = make_neighbor_table(L)
    sweep = make_sweep_fn(L, beta, UP, DOWN)
    measure_monopoles = make_measure_fn(L, UP, DOWN)

    # Thermalize
    for _ in range(n_thermalize):
        key, U_flat, _ = sweep(key, U_flat)

    # Measure
    measurements = []
    for i in range(n_sweeps):
        key, U_flat, acc = sweep(key, U_flat)
        rho = measure_monopoles(U_flat)
        measurements.append(float(rho))

    avg_rho = sum(measurements) / len(measurements)
    print(f"  ✅ Monopole Density: {avg_rho:.4e}")
    return avg_rho

# --- Main Scan Loop ---
CONFIG = {
    "betas": [5.2, 5.4, 5.6],
    "Ls": [8, 12],
    "sweeps": 200,
    "thermalize": 50
}

results = []
print("🚀 Starting Monopole Scan...")

for L in CONFIG["Ls"]:
    for beta in CONFIG["betas"]:
        rho = run_simulation(L, beta, CONFIG["sweeps"], CONFIG["thermalize"])
        results.append({"L": L, "beta": beta, "rho": rho})

print("\n📊 Summary:")
print(json.dumps(results, indent=2))

🚀 Starting Monopole Scan...
▶ Running L=8^4, Beta=5.2...
  ✅ Monopole Density: 4.1870e-04
▶ Running L=8^4, Beta=5.4...
  ✅ Monopole Density: 2.0976e-04
▶ Running L=8^4, Beta=5.6...
  ✅ Monopole Density: 1.3021e-04
▶ Running L=12^4, Beta=5.2...
  ✅ Monopole Density: 2.5471e-04
▶ Running L=12^4, Beta=5.4...
  ✅ Monopole Density: 2.0070e-04
▶ Running L=12^4, Beta=5.6...
  ✅ Monopole Density: 2.0359e-04

📊 Summary:
[
  {
    "L": 8,
    "beta": 5.2,
    "rho": 0.00041870118542647105
  },
  {
    "L": 8,
    "beta": 5.4,
    "rho": 0.0002097574929757684
  },
  {
    "L": 8,
    "beta": 5.6,
    "rho": 0.00013020833705013502
  },
  {
    "L": 12,
    "beta": 5.2,
    "rho": 0.0002547099964749577
  },
  {
    "L": 12,
    "beta": 5.4,
    "rho": 0.00020069765346761415
  },
  {
    "L": 12,
    "beta": 5.6,
    "rho": 0.0002035911710390792
  }
]
